# AI Bootcamp — Section 7, Local Practical 1
## Evaluation, Observability & Governance with Ollama + Qwen3

**No AWS / no paid API.** This is a **local equivalent** of part of Section 7, not the original AgentCore deployment/LLM-as-a-Judge/Bedrock Guardrails lab.

You will: (1) connect to Ollama; (2) define a tiny *fictional* evaluation set; (3) validate an approval policy; (4) run live Qwen3 requests; (5) score accuracy, latency and failures; (6) save JSON/Markdown evidence. Traces are simple local event logs, **not OpenTelemetry exports**.

Use the already installed **qwen3:1.7b** model. Run **code cells 1 → 8** in order with `Shift+Enter`. No private documents or credentials are sent.

## Cell 1 — Check local Ollama & model

In [ ]:
# Cell 1 — Standard-library only; no pip installs needed
import json, sys, time, uuid, re, statistics
from pathlib import Path
from urllib.request import urlopen, Request
from urllib.error import URLError, HTTPError

BASE_URL = 'http://127.0.0.1:11434'
MODEL_NAME = 'qwen3:1.7b'
with urlopen(BASE_URL + '/api/tags', timeout=12) as r:
    tags = json.load(r)
installed_models = [item.get('name','') for item in tags.get('models', [])]
assert any(name == MODEL_NAME or name.startswith(MODEL_NAME + '-') for name in installed_models), f'Model {MODEL_NAME} missing. Found: {installed_models}'
print('PASS: Local Ollama is available')
print('Model:', MODEL_NAME)
print('Python:', sys.version.split()[0])

## Cell 2 — Fictional benchmark tasks & expected answers

In [ ]:
# Cell 2 — All four examples use fabricated classroom arithmetic data
EVAL_CASES = [
    {'case_id':'MATH-01', 'prompt':'240 का 25 प्रतिशत कितना है?', 'expected':60.0},
    {'case_id':'MATH-02', 'prompt':'500 का 18 प्रतिशत कितना है?', 'expected':90.0},
    {'case_id':'BIZ-01', 'prompt':'एक काल्पनिक दुकान का revenue 1200000 रुपये और खर्च 1080000 रुपये है। Operating profit कितना है?', 'expected':120000.0},
    {'case_id':'BIZ-02', 'prompt':'एक काल्पनिक दुकान का revenue 2000 रुपये और खर्च 1800 रुपये है। Operating margin प्रतिशत में कितना है?', 'expected':10.0},
]
assert len({c['case_id'] for c in EVAL_CASES}) == len(EVAL_CASES)
assert all(isinstance(c['expected'], (int,float)) for c in EVAL_CASES)
print('PASS: 4 transparent evaluation cases defined (fictional data)')

## Cell 3 — Basic tracing, numeric scoring & policy checks

In [ ]:
# Cell 3 — No personal data or full prompts are stored in traces
TRACE_EVENTS = []
RUN_RESULTS = []

def event(case_id, stage, elapsed_ms, status, **meta):
    row = {
        'trace_id': case_id,
        'span_id': uuid.uuid4().hex[:12],
        'stage': stage,
        'elapsed_ms': round(float(elapsed_ms), 2),
        'status': status,
        **meta,
    }
    TRACE_EVENTS.append(row)
    return row

def approval_policy(action, rupees, human_approved=False):
    # Simulated approval gate only — no real fund transfers or refunds
    if action not in {'refund','money_transfer','read_catalog'}:
        return False
    if action == 'read_catalog':
        return True
    if rupees >= 1000 and not human_approved:
        return False
    return True

def parse_answer_number(text):
    # Score only the last explicitly printed number in a brief model answer
    normalized = re.sub(r'(?<=\d),(?=\d)', '', text or '')
    found = re.findall(r'(?<![\w.])-?\d+(?:\.\d+)?', normalized)
    return float(found[-1]) if found else None

print('PASS: Trace collection, grader and local approval policy defined')

## Cell 4 — Offline governance tests (no model call)

In [ ]:
# Cell 4 — Sensitive decisions blocked until human approval
POLICY_TESTS = [
    ('unapproved refund 1500', not approval_policy('refund', 1500, False)),
    ('approved refund 1500', approval_policy('refund', 1500, True)),
    ('unknown action denied', not approval_policy('delete_customer_database', 0, True)),
    ('read-only catalog permitted', approval_policy('read_catalog', 0, False)),
]
assert all(ok for _, ok in POLICY_TESTS), POLICY_TESTS
for label, ok in POLICY_TESTS:
    print('PASS:' if ok else 'FAIL:', label)
print('PASS: All 4 simulated governance checks passed')

## Cell 5 — Call the real local Qwen3 model (4 cases)

Calls can be slow on a CPU. The notebook records *failures* rather than hiding them. This cell requires Ollama running, but no cloud services.

In [ ]:
# Cell 5 — Qwen3 response + latency; `think: False` prevents empty final answers
RUN_RESULTS.clear()
TRACE_EVENTS.clear()

def ask_local_llm(case):
    message = 'केवल अंतिम संख्या लिखो, कोई व्याख्या नहीं। प्रतिशत वाले प्रश्न में सिर्फ प्रतिशत का अंक लिखो। प्रश्न: ' + case['prompt']
    started = time.perf_counter()
    latest_error = None
    for attempt in (1, 2):
        payload = {
            'model': MODEL_NAME,
            'messages': [
                {'role':'system','content':'You are a calculator for small textbook examples. Reply with ONLY a number, no explanation.'},
                {'role':'user','content':message},
            ],
            'stream':False,
            'think':False,
            'options':{'temperature':0,'num_predict':160 if attempt == 1 else 256},
        }
        try:
            req = Request(BASE_URL + '/api/chat', data=json.dumps(payload).encode('utf-8'), headers={'Content-Type':'application/json'}, method='POST')
            with urlopen(req, timeout=150) as resp:
                data = json.load(resp)
            answer = (data.get('message') or {}).get('content', '').strip()
            if answer:
                elapsed = (time.perf_counter()-started)*1000
                event(case['case_id'], 'ollama.chat', elapsed, 'ok', attempt=attempt, model=MODEL_NAME)
                return {'case_id':case['case_id'],'expected':case['expected'],'answer':answer,'latency_ms':round(elapsed,1),'error':None}
            latest_error = 'Empty Ollama content'
        except Exception as exc:
            latest_error = f'{type(exc).__name__}: {str(exc)[:130]}'
    elapsed = (time.perf_counter()-started)*1000
    event(case['case_id'], 'ollama.chat', elapsed, 'error', error_type=(latest_error or 'Unknown')[:80])
    return {'case_id':case['case_id'],'expected':case['expected'],'answer':'','latency_ms':round(elapsed,1),'error':latest_error}

for case in EVAL_CASES:
    result = ask_local_llm(case)
    RUN_RESULTS.append(result)
    print(f"{result['case_id']}: {result['answer'][:70] or '[EMPTY]'} | {result['latency_ms']:.0f}ms | {result['error'] or 'ok'}")

assert len(RUN_RESULTS) == len(EVAL_CASES)
print('PASS: Completed 4 local Qwen3 requests; failures (if any) were recorded')

## Cell 6 — Evaluate correctness and latency

In [ ]:
# Cell 6 — A completed evaluation is not automatically 100% accurate
assert len(RUN_RESULTS) == len(EVAL_CASES), 'Run Cell 5 first'
for rec in RUN_RESULTS:
    actual = parse_answer_number(rec['answer'])
    correct = rec['error'] is None and actual is not None and abs(actual-rec['expected']) < 1e-6
    rec['parsed_number'] = actual
    rec['correct'] = bool(correct)
    event(rec['case_id'], 'score.numeric', 0, 'pass' if correct else 'fail', expected=rec['expected'], observed=actual)

count = len(RUN_RESULTS)
correct_count = sum(r['correct'] for r in RUN_RESULTS)
latencies = [r['latency_ms'] for r in RUN_RESULTS]
SUMMARY = {
    'model': MODEL_NAME,
    'cases': count,
    'correct': correct_count,
    'accuracy_percent': round(100*correct_count/count, 1),
    'mean_latency_ms': round(statistics.mean(latencies), 1),
    'max_latency_ms': round(max(latencies), 1),
    'errors': sum(bool(r['error']) for r in RUN_RESULTS),
    'governance_tests_passed': sum(ok for _,ok in POLICY_TESTS),
}
for rec in RUN_RESULTS:
    print(('CORRECT' if rec['correct'] else 'INCORRECT'), rec['case_id'], '| expected:', rec['expected'], '| actual:', rec['parsed_number'])
print('\nEvaluation metrics:', SUMMARY)
print('PASS: Evaluation metrics computed — accuracy can be lower than 100%')

## Cell 7 — Inspect trace spans, governance and error counts

In [ ]:
# Cell 7 — Minimal local span audit (not an OpenTelemetry SDK export)
assert 'SUMMARY' in globals(), 'Run Cell 6 first'
assert len(TRACE_EVENTS) == len(EVAL_CASES)*2
assert all(e['trace_id'] and e['span_id'] for e in TRACE_EVENTS)
print('Model-call spans:', sum(e['stage']=='ollama.chat' for e in TRACE_EVENTS))
print('Evaluation spans:', sum(e['stage']=='score.numeric' for e in TRACE_EVENTS))
print('Failed evaluations:', SUMMARY['cases']-SUMMARY['correct'])
print('Governance policy checks:', SUMMARY['governance_tests_passed'], '/', len(POLICY_TESTS))
print('PASS: Trace/audit records complete (metadata only; no private prompts saved)')

## Cell 8 — Save evaluation report and trace evidence

In [ ]:
# Cell 8 — Save three files in Jupyter's current folder
assert 'SUMMARY' in globals() and len(TRACE_EVENTS) == len(EVAL_CASES)*2, 'Run Cells 5–7 first'
results_path = Path.cwd() / 'section7_eval_results.json'
trace_path = Path.cwd() / 'section7_trace_events.json'
report_path = Path.cwd() / 'section7_eval_report.md'
results_path.write_text(json.dumps({'summary':SUMMARY,'results':RUN_RESULTS},ensure_ascii=False,indent=2),encoding='utf-8')
trace_path.write_text(json.dumps(TRACE_EVENTS,ensure_ascii=False,indent=2),encoding='utf-8')
lines = [
    '# Section 7 — Local Qwen3 evaluation report',
    '',
    'This is a **local classroom evaluation**, not AWS AgentCore Evaluations or production validation.',
    '',
    f"- Model: `{SUMMARY['model']}`",
    f"- Correct answers: **{SUMMARY['correct']} / {SUMMARY['cases']}**",
    f"- Accuracy: **{SUMMARY['accuracy_percent']}%**",
    f"- Mean latency: **{SUMMARY['mean_latency_ms']} ms**",
    f"- Max latency: **{SUMMARY['max_latency_ms']} ms**",
    f"- Local API errors: **{SUMMARY['errors']}**",
    f"- Governance tests passed: **{SUMMARY['governance_tests_passed']} / 4**",
    '',
    '| Case | Expected | Parsed | Correct | Latency (ms) |',
    '|---|---:|---:|:---:|---:|',
]
for r in RUN_RESULTS:
    lines.append(f"| {r['case_id']} | {r['expected']} | {r['parsed_number']} | {'Yes' if r['correct'] else 'No'} | {r['latency_ms']} |")
lines += ['', 'Review every failed case. A PASS in the notebook means the **evaluation pipeline** ran, not that the model has perfect accuracy.', '']
report_path.write_text('\n'.join(lines),encoding='utf-8')
print('PASS: Section 7 local evaluation and trace reports saved')
print('Saved:', results_path.name, trace_path.name, report_path.name)

---
**Completion rule:** A local successful run is demonstrated by eight code cells executing and three saved files. **Model quality** is the percentage from Cell 6, not the PASS count. Original AWS AgentCore Evaluation, OTel / Langfuse / LangSmith / CloudWatch and Bedrock governance labs remain separate from this local exercise.

Original Section 7 reference: https://scaler-content.github.io/evals-observability-governance/